In [17]:
import pandas as pd
import numpy as np
import uuid
from datetime import datetime, timedelta, timezone
import os

In [63]:
df_tx = pd.read_csv('/content/drive/MyDrive/FinCore/transactions.csv')
df_cust = pd.read_csv('/content/drive/MyDrive/FinCore/customers.csv')
df_inv = pd.read_csv('/content/drive/MyDrive/FinCore/investments.csv')

BASE = "/content/drive/MyDrive/FinCore/"

In [39]:
df_inv.head(20)

,position_id,customer_id,product_type,product_name,invested_amount,current_value,purchase_date,maturity_date,annual_rate,risk_level
0,17fe113d-e972-4d6d-9772-af3e41e29b86,6147,CDB,CDB BTG 115% CDI,13122.26,15095.85,2023-01-01,2024-10-20,119.50,baixo
1,def3b464-79e4-41fb-a382-9695d089d78c,5485,LCA,LCA Caixa 96% CDI,12822.36,12569.76,2023-01-11,2024-03-12,91.27,baixo
2,94bbc965-d31f-416a-aa60-c7ab173b5f1d,384,CDB,CDB XP 108% CDI,4031.10,3897.67,2023-04-13,2024-07-21,116.63,baixo
3,d90679dc-5ba0-4af7-a620-688b0e03c768,50,Ação,BBDC4,2617.78,3171.18,2022-08-04,NaN,30.70,alto
4,4479dd6e-b86a-4e4a-8f49-896656fe68e2,5331,LCI,LCI Santander 93% CDI,238.60,253.97,2022-08-14,2024-09-08,94.20,baixo
5,d150589b-0e55-454a-aeb6-c30e82e75b78,6287,Tesouro,Tesouro Selic 2026,442.92,516.75,2023-03-21,2024-03-23,11.55,baixo
6,4f3e515f-e4b6-4623-86cc-3861a2d94d55,6660,LCA,LCA Caixa 96% CDI,1974.22,2355.84,2023-02-19,2025-11-14,92.57,baixo
7,2b0e8fd8-a972-4d01-8895-1bb14511d8ec,5899,LCA,LCA Banco do Brasil 94% CDI,5908.53,6117.10,2023-01-14,2023-08-23,93.92,baixo
8,263cd4ca-22e9-4153-a487-f13e7bcecae4,6109,LCI,LCI Itaú 95% CDI,930.21,1091.97,2022-01-30,2024-10-27,90.53,baixo
9,9b2d0f80-6e8d-406c-886b-a62494a8f393,9904,Tesouro,Tesouro Prefixado 2027,1441.16,1377.17,2023-05-06,2025-06-19,12.69,baixo


# Task 01: Ingestão e Camada Bronze

## Organizar a estrutura de pastas
>Ferramenta: Python (os, pathlib)

In [41]:
from pathlib import Path

# Estrutura de pastas do Data Lake
folders = [
    "/content/drive/MyDrive/FinCore/data-lake/bronze/transactions",
    "/content/drive/MyDrive/FinCore/data-lake/bronze/customers",
    "/content/drive/MyDrive/FinCore/data-lake/bronze/investments",
    "/content/drive/MyDrive/FinCore/data-lake/logs"
]

for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("✅ Estrutura de pastas criada!")

✅ Estrutura de pastas criada!


## Adicionar metadados de ingestão
> Ferramenta: Pandas

In [42]:
def add_ingestion_metadata(df, source_file):
    df = df.copy()
    df['_ingestion_timestamp'] = datetime.now(timezone.utc).isoformat()
    df['_source_file']         = source_file
    df['_ingestion_date']      = datetime.now(timezone.utc).strftime('%Y-%m-%d')
    return df

df_tx = add_ingestion_metadata(df_tx,'transactions.csv')
df_cust = add_ingestion_metadata(df_cust,'customers.csv')
df_inv = add_ingestion_metadata(df_inv,'investments.csv')

print("✅ Metadados adicionados!")
print(df_tx[['transaction_id', '_ingestion_timestamp', '_source_file']].head(2))

✅ Metadados adicionados!
                         transaction_id              _ingestion_timestamp  \
0  6c1d797f-2c64-4122-8152-9e6c0af51edd  2026-02-26T14:45:12.936655+00:00   
1  5ad3cc6c-d4d8-422e-8ddd-a7f134e1adae  2026-02-26T14:45:12.936655+00:00   

       _source_file  
0  transactions.csv  
1  transactions.csv  


## Extrair ano e mês para particionamento
>Ferramenta: Pandas

In [44]:
# Converte datas com coerce (erros viram NaT em vez de quebrar)
df_tx['_parsed_date'] = pd.to_datetime(
    df_tx['transaction_date'],
    infer_datetime_format=True,
    errors='coerce'
)

df_tx['_year']  = df_tx['_parsed_date'].dt.year.fillna(9999).astype(int)
df_tx['_month'] = df_tx['_parsed_date'].dt.month.fillna(99).astype(int)

print("✅ Partições extraídas!")
print(df_tx.groupby(['_year','_month']).size().reset_index(name='rows'))

✅ Partições extraídas!
    _year  _month    rows
0    2024       1    5466
1    2024       2    5496
2    2024       3    5512
3    2024       4    5491
4    2024       5    5489
5    2024       6    5393
6    2024       7    5475
7    2024       8    5590
8    2024       9    5451
9    2024      10    5498
10   2024      11    5563
11   2024      12    5287
12   9999      99  434289


/tmp/ipython-input-201/3540416561.py:2: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_tx['_parsed_date'] = pd.to_datetime(


## Salvar em Parquet particionado
>Ferramenta: Pandas + PyArrow

In [45]:
# Transactions — particionado por ano e mês
df_tx.to_parquet(
    "data-lake/bronze/transactions",
    partition_cols=['_year', '_month'],
    index=False,
    engine='pyarrow'
)

# Customers e Investments — arquivo único (sem partição temporal)
df_cust.to_parquet(
    "/content/drive/MyDrive/FinCore/data-lake/bronze/customers/customers.parquet",
    index=False,
    engine='pyarrow'
)

df_inv.to_parquet(
    "/content/drive/MyDrive/FinCore/data-lake/bronze/investments/investments.parquet",
    index=False,
    engine='pyarrow'
)

print("✅ Parquet salvo!")

✅ Parquet salvo!


In [46]:
for root, dirs, files in os.walk("/content/drive/MyDrive/FinCore/data-lake/bronze"):
    level = root.replace("/content/drive/MyDrive/FinCore/data-lake/bronze", '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for file in files:
        size = os.path.getsize(os.path.join(root, file)) / 1024
        print(f'{indent}  {file}  ({size:.0f} KB)')

bronze/
  investments/
    investments.parquet  (1536 KB)
  customers/
    customers.parquet  (312 KB)
  transactions/
    _year=9999/
      _month=99/
        4f8fad06fb5c440f8860ad8efd5afb1b-0.parquet  (21336 KB)
    _year=2024/
      _month=3/
        4f8fad06fb5c440f8860ad8efd5afb1b-0.parquet  (342 KB)
      _month=9/
        4f8fad06fb5c440f8860ad8efd5afb1b-0.parquet  (339 KB)
      _month=1/
        4f8fad06fb5c440f8860ad8efd5afb1b-0.parquet  (339 KB)
      _month=4/
        4f8fad06fb5c440f8860ad8efd5afb1b-0.parquet  (340 KB)
      _month=10/
        4f8fad06fb5c440f8860ad8efd5afb1b-0.parquet  (342 KB)
      _month=6/
        4f8fad06fb5c440f8860ad8efd5afb1b-0.parquet  (335 KB)
      _month=7/
        4f8fad06fb5c440f8860ad8efd5afb1b-0.parquet  (340 KB)
      _month=11/
        4f8fad06fb5c440f8860ad8efd5afb1b-0.parquet  (345 KB)
      _month=2/
        4f8fad06fb5c440f8860ad8efd5afb1b-0.parquet  (341 KB)
      _month=8/
        4f8fad06fb5c440f8860ad8efd5afb1b-0.parquet  (346 K

## Gerar o log de ingestão
>Ferramenta: Python (json)

In [64]:
import json

def build_ingestion_log(df, source_file, output_path):
    return {
        "source_file":           source_file,
        "ingestion_timestamp":   datetime.utcnow().isoformat(),
        "status":                "SUCCESS",
        "row_count":             len(df),
        "column_count":          len(df.columns),
        "null_summary": {
            col: int(df[col].isna().sum())
            for col in df.columns
            if not col.startswith('_')
        }
    }

log = {
    "pipeline_run": datetime.utcnow().isoformat(),
    "datasets": [
        build_ingestion_log(df_tx,    'transactions.csv',  'bronze/transactions'),
        build_ingestion_log(df_cust,  'customers.csv',     'bronze/customers'),
        build_ingestion_log(df_inv,   'investments.csv',   'bronze/investments'),
    ]
}

log_path = f"/content/drive/MyDrive/FinCore/data-lake/logs/ingestion_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.json"
with open(log_path, 'w', encoding='utf-8') as f:
    json.dump(log, f, indent=2, ensure_ascii=False)

print(f"✅ Log salvo em: {log_path}")
print(json.dumps(log, indent=2, ensure_ascii=False)[:800])  # Preview

✅ Log salvo em: /content/drive/MyDrive/FinCore/data-lake/logs/ingestion_20260226_150859.json
{
  "pipeline_run": "2026-02-26T15:08:59.297298",
  "datasets": [
    {
      "source_file": "transactions.csv",
      "ingestion_timestamp": "2026-02-26T15:08:59.297344",
      "status": "SUCCESS",
      "row_count": 500000,
      "column_count": 10,
      "null_summary": {
        "transaction_id": 0,
        "customer_id": 5051,
        "transaction_date": 0,
        "amount": 0,
        "transaction_type": 0,
        "merchant_category": 49838,
        "status": 0,
        "channel": 0,
        "is_fraud_flag": 74557,
        "installments": 350039
      }
    },
    {
      "source_file": "customers.csv",
      "ingestion_timestamp": "2026-02-26T15:08:59.484110",
      "status": "SUCCESS",
      "row_count": 10000,
      "column_count": 11,
      "null_summary": {
        "customer_id


/tmp/ipython-input-201/1860872744.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "pipeline_run": datetime.utcnow().isoformat(),
/tmp/ipython-input-201/1860872744.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingestion_timestamp":   datetime.utcnow().isoformat(),
/tmp/ipython-input-201/1860872744.py:26: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  log_path = f"/content/drive/MyDrive/FinCore/data-lake/logs/ingestion_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.json"


## Validação final: ler de volta o Parquet
>Ferramenta: Pandas

In [65]:
df_check = pd.read_parquet(BASE + "/data-lake/bronze/transactions/_year=2024/_month=1")

print(f"✅ Leitura OK — {len(df_check):,} transações de Jan/2024")
print(f"   Colunas: {list(df_check.columns)}")
print(f"   Tamanho em memória: {df_check.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

✅ Leitura OK — 5,466 transações de Jan/2024
   Colunas: ['transaction_id', 'customer_id', 'transaction_date', 'amount', 'transaction_type', 'merchant_category', 'status', 'channel', 'is_fraud_flag', 'installments', '_ingestion_timestamp', '_source_file', '_ingestion_date', '_parsed_date']
   Tamanho em memória: 3.4 MB


# Task 02 — Qualidade de Dados e Camada Silver

## Ler os dados da Bronze

In [66]:
import pandas as pd
import glob
from pathlib import Path

# transactions — lê todas as partições
parquet_files = glob.glob(BASE + "data-lake/bronze/transactions/**/*.parquet", recursive=True)
df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)

# customers e investments — arquivo único, leitura direta
df_cust = pd.read_parquet(BASE + "data-lake/bronze/customers/customers.parquet")
df_inv  = pd.read_parquet(BASE + "data-lake/bronze/investments/investments.parquet")

print(f"✅ Bronze carregada!")
print(f"   transactions: {len(df):,} linhas")
print(f"   customers:    {len(df_cust):,} linhas")
print(f"   investments:  {len(df_inv):,} linhas")

✅ Bronze carregada!
   transactions: 500,000 linhas
   customers:    10,000 linhas
   investments:  25,000 linhas


## Diagnóstico antes de qualquer limpeza

In [67]:
def quality_diagnosis(df, name):
    print(f"\n{'='*50}")
    print(f"📋 DIAGNÓSTICO — {name}")
    print(f"{'='*50}")
    print(f"Linhas: {len(df):,} | Colunas: {df.shape[1]}")

    print("\n--- Nulos por coluna ---")
    nulls = df.isna().sum()
    nulls_pct = (nulls / len(df) * 100).round(2)
    diag = pd.DataFrame({'nulos': nulls, 'pct': nulls_pct})
    print(diag[diag['nulos'] > 0].to_string())

    print("\n--- Duplicatas ---")
    if 'transaction_id' in df.columns:
        dups = df.duplicated(subset='transaction_id').sum()
        print(f"  transaction_id duplicados: {dups:,} ({dups/len(df)*100:.2f}%)")

    print("\n--- Valores únicos (colunas categóricas) ---")
    for col in ['transaction_type', 'status', 'channel']:
        if col in df.columns:
            print(f"  {col}: {sorted(df[col].dropna().unique())}")

    return {'linhas': len(df), 'nulos': diag[diag['nulos'] > 0].to_dict()}

diag_antes = quality_diagnosis(df, 'transactions — ANTES DA LIMPEZA')


📋 DIAGNÓSTICO — transactions — ANTES DA LIMPEZA
Linhas: 500,000 | Colunas: 14

--- Nulos por coluna ---
                    nulos    pct
customer_id          5051   1.01
merchant_category   49838   9.97
is_fraud_flag       74557  14.91
installments       350039  70.01
_parsed_date       434289  86.86

--- Duplicatas ---
  transaction_id duplicados: 1,995 (0.40%)

--- Valores únicos (colunas categóricas) ---
  transaction_type: ['BOLETO', 'Boleto', 'CARTAO', 'Cartão', 'DOC', 'PIX', 'TED', 'cartão', 'pix', 'ted']
  status: ['completed', 'failed', 'pending', 'reversed']
  channel: ['app', 'atm', 'branch', 'web']


## Remover duplicatas

In [68]:
linhas_antes = len(df)

# Ordena pelo timestamp de ingestão para manter o registro mais recente
df = df.sort_values('_ingestion_timestamp', ascending=False)

# Remove duplicatas mantendo a primeira ocorrência (mais recente)
df = df.drop_duplicates(subset='transaction_id', keep='first')

duplicatas_removidas = linhas_antes - len(df)
print(f"✅ Duplicatas removidas: {duplicatas_removidas:,}")
print(f"   Antes: {linhas_antes:,} | Depois: {len(df):,}")

✅ Duplicatas removidas: 1,995
   Antes: 500,000 | Depois: 498,005


## Padronizar datas

In [69]:
def parse_mixed_dates(series):
    """Tenta múltiplos formatos de data e retorna NaT para inválidos."""
    parsed = pd.to_datetime(series, infer_datetime_format=True, errors='coerce')

    # Tenta formato BR para os que falharam
    mask_failed = parsed.isna() & series.notna()
    if mask_failed.sum() > 0:
        parsed[mask_failed] = pd.to_datetime(
            series[mask_failed],
            format='%d/%m/%Y %H:%M',
            errors='coerce'
        )
    return parsed

df['transaction_date_clean'] = parse_mixed_dates(df['transaction_date'])

datas_invalidas = df['transaction_date_clean'].isna().sum()
print(f"✅ Datas padronizadas!")
print(f"   Datas inválidas (virou NaT): {datas_invalidas:,}")
print(f"   Exemplo antes: {df['transaction_date'].iloc[0]}")
print(f"   Exemplo depois: {df['transaction_date_clean'].iloc[0]}")

✅ Datas padronizadas!
   Datas inválidas (virou NaT): 0
   Exemplo antes: 2024-12-14T00:00:00
   Exemplo depois: 2024-12-14 00:00:00


/tmp/ipython-input-201/1599667559.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(series, infer_datetime_format=True, errors='coerce')


## Normalizar transaction_type

In [ ]:
import unicodedata

def normalize_text(s):
    """Remove acentos e converte para uppercase."""
    if pd.isna(s):
        return s
    s = unicodedata.normalize('NFKD', str(s))
    s = s.encode('ascii', 'ignore').decode('ascii')
    return s.strip().upper()

# Mapa de padronização
type_map = {
    'PIX': 'PIX', 'TED': 'TED', 'DOC': 'DOC',
    'BOLETO': 'BOLETO', 'CARTAO': 'CARTAO'
}

df['transaction_type_clean'] = (
    df['transaction_type']
    .apply(normalize_text)
    .map(type_map)
)

print("✅ transaction_type normalizado!")
print(df['transaction_type_clean'].value_counts())